# v4 파이프라인 전체 — 질문 하나가 Step 1~4 를 지나며 무엇이 되는가

질문을 넣으면 단계마다 **입력·파라미터·출력**을 표로 보여 주고, 마지막에 **찾은 View(잘라 낸 표)** 를 그대로 보여 준다.
계산은 API·CLI 와 같은 backend 함수, 화면 출력은 `notebooks/v4/e2e_helpers.py`.

| 단계 | 하는 일 | 함수 (`backend/…`) | Swagger |
|---|---|---|---|
| ⓪ 준비 | 표 색인 · CSV 값 사전 | `rag/ingestion/table_text` · `rag/csv_index` | `POST /api/v1/rag/chunk-and-index` · `/api/v1/csv-index/build` |
| ① 쿼리 (Step 2) | 재작성 → 동의어·접지 → 시점 규칙·되묻기 → 서브쿼리 → 검색 문장·view_bundle | `rag/query_analyzer/v4.analyze` | `POST /api/v1/rag/v4/analyze` · `/clarify` |
| ② 검색 (Step 3) | 검색 단위마다 구성별 후보 표 (단계별 순위) | `rag/retrieval/factory.get_pipeline(…).search` | `POST /api/v1/retrieval-lab/search` · `/compare` |
| ③ View 묶음 (Step 4) | 단위마다 상위 표를 슬롯대로 잘라 View | `rag/view.build_viewset` | `POST /api/v1/rag/v4/views` |
| ④ 충돌·선택 (Step 4) | 같은 교차점 값 비교 → 선택지 → 선택 | `detect` · `apply_selections` | `POST /api/v1/rag/v4/views/select` |
| ⑤ 최종 View | 답변 모듈로 넘길 Record (+ 선택: HCX 답변) | `agent/light_agent.answer_from_records` | `POST /api/v1/agent/answer` |

**쓰는 법**: `설정` 셀에서 질문·검색 구성·되묻기/선택 방식을 바꾸고 위에서부터 실행 →
`summary_table(R)` 로 전체, `show(R[질문])` 으로 한 질문을 단계별로.

> 저장소 루트에서 실행. 실제 데이터·HCX 를 쓰면 질문마다 HCX 1회(쿼리 분석) + 임베딩(캐시).
> 데이터·HCX 없이 구조만 보려면 `V4_NB_FAKE=1` 환경변수 (가짜 데이터 + HCX 대역 + 충돌 재현용 표).

## 0. 환경

In [1]:
import os, sys, json, time
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parents[1]   # notebooks/v4 에서 열어도
sys.path[:0] = [str(ROOT), str(ROOT / "notebooks" / "v4")]; os.chdir(ROOT)
USE_FAKE = os.getenv("V4_NB_FAKE", "0") == "1"
if USE_FAKE:                                   # 가짜 KOSIS 데이터 + 표 15개 + 충돌 재현 표 2개, HCX 대역
    import tempfile
    from tests import fake_kosis
    from tests.v4.test_retrieval_lab import _extra_tables
    from tests.v4.test_view_v4 import build_conflict_data
    tmp = Path(tempfile.mkdtemp()) / "storage"
    fake_kosis.TABLES[:] = fake_kosis.TABLES + _extra_tables()
    build_conflict_data(tmp / "kosis_data")
    for k in ("KOSIS_DATA_DIR", "KOSIS_DATA", "VIEW_OUT_DIR", "TABLE_INDEX_DIR", "VIEW_EMBED_CACHE", "CSV_INDEX_DIR", "RUNS_DIR"):
        os.environ[k] = ""
    os.environ["STORAGE_DIR"], os.environ["HCX_API_KEY"] = str(tmp), "test"
    import backend.core.llm as llm, backend.rag.retrieval.dense as dense
    llm._post, dense._post = fake_kosis.fake_hcx_post, fake_kosis.fake_dense_post

import pandas as pd
from IPython.display import display
pd.set_option("display.max_colwidth", 70); pd.set_option("display.max_rows", 60)
from backend.core.config import KOSIS_DATA_DIR, TABLE_INDEX_DIR, CSV_INDEX_DIR, RUNS_DIR
from e2e_helpers import run_one, show, summary_table, choose, show_options, show_records, ok_configs
print("데이터:", KOSIS_DATA_DIR, "\n표 색인:", TABLE_INDEX_DIR, "\nCSV 값 사전:", CSV_INDEX_DIR, "\n실행 기록:", RUNS_DIR,
      "\n가짜 데이터:", USE_FAKE)

데이터: C:\Project\Llion\project\t4core-stat-agent\storage\kosis_data 
표 색인: C:\Project\Llion\project\t4core-stat-agent\storage\indices\table_index 
CSV 값 사전: C:\Project\Llion\project\t4core-stat-agent\storage\csv_index 
실행 기록: C:\Project\Llion\project\t4core-stat-agent\storage\runs 
가짜 데이터: False


## ⓪ 준비 — 표 색인 · CSV 값 사전

이미 있으면 건너뜀. `REBUILD=True` 면 다시 만든다 (CSV 값 사전은 바뀐 표만 다시 스캔).

In [2]:
REBUILD = False
from backend.rag.ingestion import meta_profile, table_text
from backend.rag.csv_index import build_index, load_index
if REBUILD or USE_FAKE or not (Path(TABLE_INDEX_DIR) / "tokens.json").exists():
    meta_profile.build_all(); table_text.build_index(progress=print)
if REBUILD or USE_FAKE or not (Path(CSV_INDEX_DIR) / "values.parquet").exists():
    build_index(scope="all", log=print)
idx = load_index()
print("표 색인:", json.loads((Path(TABLE_INDEX_DIR) / "meta.json").read_text(encoding="utf-8")))
print("CSV 값 사전:", {k: idx.summary.get(k) for k in ("n_tables", "n_values", "by_slot")})

표 색인: {'n_tables': 1092, 'created': '2026-11-03T13:45:14'}
CSV 값 사전: {'n_tables': 1092, 'n_values': 390757, 'by_slot': {'지역': 323141, '분류': 32362, '연령': 14502, '국적': 10072, '항목': 6370, '열': 2513, '표': 1092, '성별': 705}}


## 설정 — 여기만 바꿔서 실험

| 키 | 뜻 |
|---|---|
| `QUESTIONS` | 실험 질문 (가짜 데이터면 아래 예시가 충돌·추이·성별연령·서브쿼리·되묻기·범위 밖을 하나씩 보여 줌) |
| `today` | 기준 날짜 ('작년'·'최근' 해석) |
| `query_overrides` | `configs/query_v4.yaml` 값 일부 (예: `{"base_year": 2025}`, `{"conflict_rel_tol": 0.01}`, `{"view_keep_unusable": False}`) |
| `retrieval_configs` | ② 에서 나란히 볼 검색 구성 (`configs/retrieval/*.yaml`). 설치 안 된 구성은 자동으로 뺌 |
| `view_config` | ③ View 를 만들 때 쓸 검색 구성 (`None` = 기존 검색 = baseline) |
| `n_views` | 단위마다 View 를 만들어 비교할 표 수 |
| `clarify` | 되묻기 응답: `"first"`(첫 후보) · `"ask"`(input) · `{질문: {슬롯: [값]}}` · `None`(멈춤) |
| `select` | 충돌 선택: `"first"` · `"ask"` · `{질문: "opt-2"}` · `None`(선택 대기로 두고 아래 '선택' 셀에서) |

In [13]:
QUESTIONS = [
    "2025년 전국에서 혼자 사는 가구는 몇 가구로 집계됐지?",                     # 같은 교차점 값이 다른 표 → 충돌·선택 (가짜 데이터)
    "2024년에서 2025년 사이 다문화가구는 전국적으로 몇 가구 늘었나?",          # 추이 (시점 규칙)
    "2016년과 2025년의 전국 한부모가구 차이를 가구 수로 말해 줘.",             # 성별·연령 슬롯
    "2025년 전국 주택은 모두 몇 호인지 알려줘.",         # 함께 가진 표 없음 → 서브쿼리 2개
    "2025년에 태어난 신생아가 전국팔도에서 몇 명이었어?",               # 지역 없음 → 되묻기 + '요즘' → 2026
    "2025 혼인껀수랑 2024 혼인 건수 차이",
    "2023년 남녀 평균초혼연령 차이는 몇 살인지 알려줘.",
    "2023년 전남 신안군 외국인주민 전체가 몇 명인지 찾아줘.",
    "2025년 국내 등록외국인 가운데 베트남 국적은 몇 명이야?",
    "2026년 9월 한 달간 전국에서 이사한 사람 수는?",
    "올해 우리나라 인구 전망치를 중위 가정으로 보면 몇 명이야?",
    "2026년 전체 인구에서 65세 이상이 차지하는 비중은 몇 퍼센트로 예상돼?",
    "2035년 한 집에서 솔로 가구는 얼마나 될 것으로 예상돼?",
    "2024년 해외에서 들어온 한국인과 해외로 나간 한국인의 차이가 얼마야?",
    "2026년 9월 말 전국 주민등록 인구가 얼마로 집계됐어?",
]
CFG = {
    "today": "2026-11-09",
    "query_overrides": {"csv_scope": "all"} if USE_FAKE else {},
    "retrieval_configs": ["baseline", "bm25_only", "kiwi_lexicon", "hybrid_rrf_hcx"],
    "view_config": None,
    "n_views": 3,
    "clarify": "first",
    "select": None,
    "max_view_rows": 15,
}
RUN_ANSWER = True                 # ⑤ 에서 HCX 답변까지 (실제 HCX 키 필요)

## 실행 — 질문 전체

In [14]:
R = {}
for q in QUESTIONS:
    print("▶", q)
    R[q] = run_one(q, CFG)
summary_table(R)

▶ 2025년 전국에서 혼자 사는 가구는 몇 가구로 집계됐지?
▶ 2024년에서 2025년 사이 다문화가구는 전국적으로 몇 가구 늘었나?
▶ 2016년과 2025년의 전국 한부모가구 차이를 가구 수로 말해 줘.
▶ 2025년 전국 주택은 모두 몇 호인지 알려줘.
▶ 2025년에 태어난 신생아가 전국팔도에서 몇 명이었어?
▶ 2025 혼인껀수랑 2024 혼인 건수 차이
▶ 2023년 남녀 평균초혼연령 차이는 몇 살인지 알려줘.
▶ 2023년 전남 신안군 외국인주민 전체가 몇 명인지 찾아줘.
▶ 2025년 국내 등록외국인 가운데 베트남 국적은 몇 명이야?
▶ 2026년 9월 한 달간 전국에서 이사한 사람 수는?
  (분류 판정 규칙 변경 → 다시 정규화) 358_DT_358005_011: [('특성별', 'sex', 'other')]
  (분류 판정 규칙 변경 → 다시 정규화) 182_DT_182001_A032_2025: [('특성별', 'sex', 'other')]
▶ 올해 우리나라 인구 전망치를 중위 가정으로 보면 몇 명이야?
▶ 2026년 전체 인구에서 65세 이상이 차지하는 비중은 몇 퍼센트로 예상돼?
▶ 2035년 한 집에서 솔로 가구는 얼마나 될 것으로 예상돼?
▶ 2024년 해외에서 들어온 한국인과 해외로 나간 한국인의 차이가 얼마야?
▶ 2026년 9월 말 전국 주민등록 인구가 얼마로 집계됐어?


,질문,① 분석,검색 문장,단위 수,② baseline 1위,④ 충돌,최종 상태,최종 View 수,초
0,2025년 전국에서 혼자 사는 가구는 몇 가구로 집계됐지?,ok,1인가구 가구수,1,DT_1JC1516,ok,ok,3,2.66
1,2024년에서 2025년 사이 다문화가구는 전국적으로 몇 가구 늘었나?,ok,다문화가구 가구수 증감,2,DT_1JU1507 / DT_1JD1505,ok / ok,ok,6,3.37
2,2016년과 2025년의 전국 한부모가구 차이를 가구 수로 말해 줘.,ok,한부모 가구수 증감 부부+한부모,2,DT_1JU1507 / DT_1JC1517,ok / ok,ok,6,3.34
3,2025년 전국 주택은 모두 몇 호인지 알려줘.,ok,주택수,1,DT_1JU1503,ok,ok,3,2.53
4,2025년에 태어난 신생아가 전국팔도에서 몇 명이었어?,ok,영유아 출생아수 지역별 인구 영유아_계,2,DT_1B81A28 / DT_1YC1601,ok / ok,ok,6,4.05
5,2025 혼인껀수랑 2024 혼인 건수 차이,ok,혼인건수 증감 추이,1,DT_1B8000G,ok,ok,3,3.08
6,2023년 남녀 평균초혼연령 차이는 몇 살인지 알려줘.,ok,남자 여자 평균초혼연령 비교,1,DT_1B83A09,partial,ok,3,2.65
7,2023년 전남 신안군 외국인주민 전체가 몇 명인지 찾아줘.,ok,외국인주민 인구수,2,DT_1IN1503 / DT_110025_A031_A,ok / ambiguous,needs_selection,3,3.24
8,2025년 국내 등록외국인 가운데 베트남 국적은 몇 명이야?,ok,베트남 등록외국인 인구수,2,DT_1IN1503 / DT_1B040A7,ok / ok,ok,6,3.46
9,2026년 9월 한 달간 전국에서 이사한 사람 수는?,ok (되묻기 응답),이동자수,1,DT_1B26002,partial,ok,3,3.61


## 질문별 자세히 — 번호를 바꿔 가며 (`show(R[질문])`)

In [15]:
show(R[QUESTIONS[0]], CFG)

# 2025년 전국에서 혼자 사는 가구는 몇 가구로 집계됐지?  (2.66초)

## ① 쿼리 분석 (Step 2)

**상태** `ok` · 검색 문장 `1인가구 가구수` · 서브쿼리: 지표 ['가구수', '1인가구'] 를 함께 가진 표 7개 → 한 표에서 같이 자름

**1-2·1-3 슬롯 → CSV 값 (동의어·접지)**

,슬롯,원래 표현,표준 값,출처,일치,CSV 접지,가진 표 수,메모
0,시점,2025년,2025,rule,,True,358,
1,지역,전국,전국,rule,exact,True,636,
2,대상,1인가구,1인가구,csv,exact,True,90,
3,지표,가구수,가구수,csv,exact,True,61,


**View 입력 (view_bundle)** — 기존 `POST /api/v1/rag/views` 와 같은 모양

... (15 outputs left out)

**`view::101_DT_1JC1511::Y::f256ed7f-2::value`**  
인구총조사 · 단위 가구 명 · 기간 2025~2025 · value · 값 2칸 · [KOSIS](https://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1JC1511)

,시점,행정구역별(시군구),가구주의 연령별,항목,값
0,2025,전국,합계,일반가구,22495096.0
1,2025,전국,합계,평균 가구원 수,2.2


**`view::101_DT_1JD1505::Y::f256ed7f-3::value`**  
인구총조사 · 단위 가구 명 · 기간 2025~2025 · value · 값 2칸 · [KOSIS](https://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1JD1505)

,시점,행정구역별(시도),다문화가구 구성별,항목,값
0,2025,전국,계,다문화가구_계,451492.0
1,2025,전국,계,평균 가구원 수,2.8


In [16]:
show(R[QUESTIONS[3]], CFG)

# 2025년 전국 주택은 모두 몇 호인지 알려줘.  (2.53초)

## ① 쿼리 분석 (Step 2)

**상태** `ok` · 검색 문장 `주택수` · 서브쿼리: 지표(대상)가 하나 → 나누지 않음

**1-2·1-3 슬롯 → CSV 값 (동의어·접지)**

,슬롯,원래 표현,표준 값,출처,일치,CSV 접지,가진 표 수,메모
0,시점,2025년,2025,rule,,True,358,
1,지역,전국,전국,rule,exact,True,636,
2,대상,주택,주택,csv,exact,True,89,
3,지표,주택수,주택수,csv,exact,True,93,


**View 입력 (view_bundle)** — 기존 `POST /api/v1/rag/views` 와 같은 모양

... (16 outputs left out)

**`view::101_DT_1MH2090::Y::aff8df9c-2::value`**  
주택총조사 · 단위 호 · 기간 2025~2025 · value · 값 8칸 · [KOSIS](https://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1MH2090)

,시점,행정구역별(시군구),항목,값
0,2025,전국,주택_계(부가자료),23239118.0
1,2025,전국,단독주택_계,6887334.0
2,2025,전국,단독주택_일반,2644507.0
3,2025,전국,단독주택_다가구(구분거처반영),3795787.0
4,2025,전국,단독주택_영업겸용,447040.0
5,2025,전국,연립주택,544476.0
6,2025,전국,다세대주택,2306803.0
7,2025,전국,비주거용 건물 내 주택,211285.0


**`view::101_DT_1JC1505::Y::aff8df9c-3::value`**  
인구총조사 · 단위 가구 · 기간 2025~2025 · value · 값 12칸 · [KOSIS](https://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1JC1505)

,시점,행정구역별(시군구),가구원수별,항목,값
0,2025,전국,계,주택_계,21121883.0
1,2025,전국,계,주택_단독주택,6112421.0
2,2025,전국,계,주택_아파트,12263718.0
3,2025,전국,계,주택_연립주택,456445.0
4,2025,전국,계,주택_다세대주택,1983380.0
5,2025,전국,계,주택_비주거용 건물 내 주택,305919.0
6,2025,전국,계,주택 이외의 거처_계,1373213.0
7,2025,전국,계,주택 이외의 거처_오피스텔,868566.0
8,2025,전국,계,주택 이외의 거처_호텔ㆍ여관 등 숙박업소의 객실,50534.0
9,2025,전국,계,주택 이외의 거처_기숙사 및 특수사회시설,35057.0


In [17]:
show(R[QUESTIONS[4]], CFG)

# 2025년에 태어난 신생아가 전국팔도에서 몇 명이었어?  (4.05초)

## ① 쿼리 분석 (Step 2)

**상태** `ok` · 검색 문장 `영유아 출생아수 지역별 인구 영유아_계` · 서브쿼리: 지표 ['출생아수', '영유아_계'] 를 함께 가진 표가 없음 → 2개로 나눔

**1-2·1-3 슬롯 → CSV 값 (동의어·접지)**

,슬롯,원래 표현,표준 값,출처,일치,CSV 접지,가진 표 수,메모
0,시점,2025년,2025,rule,,True,358,
1,지역,전국,전국,rule,exact,True,636,
2,대상,영유아,영유아_계,contains,contains,True,1,
3,지표,출생아수,출생아수,csv,exact,True,16,
4,비교축,지역,지역별,csv,variant,True,2,


**1-5 서브쿼리**

... (27 outputs left out)

**`view::101_DT_1B28025::Y::4b5a4156-2-2::value`**  
국제인구이동통계 · 단위  · 기간 2025~2025 · value · 값 4칸 · [KOSIS](https://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1B28025)  
⚠ 항목: 지표·대상과 맞는 항목 없음 → 항목 전체

,시점,내외국인별,성별,연령별,항목,값
0,2025,계,계,계,총이동,1296102.0
1,2025,계,계,계,입국자,685242.0
2,2025,계,계,계,출국자,610860.0
3,2025,계,계,계,국제순이동,74382.0


**`view::101_DT_1SP1601::Y::4b5a4156-2-3::value`**  
인구총조사 · 단위 가구 · 기간 2025~2025 · value · 값 1칸 · [KOSIS](https://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1SP1601)  
⚠ 항목: 지표·대상과 맞는 항목 없음 → '한부모 가구_계'

,시점,행정구역별(시도),항목,값
0,2025,전국,한부모 가구_계,1477492.0


## 선택 — 충돌이 남은 질문 고르기

`choose(R, 질문, option_id)` → 선택을 반영하고 최종 View 를 보여 준다.
(Swagger: `POST /api/v1/rag/v4/views/select` 에 `{selection_token, option_id}` + 이전 응답을 `previous` 로)

In [18]:
pending = [q for q, r in R.items() if (r.get("viewset") or {}).get("pending")]
print("선택 대기:", pending)
if pending:
    show_options(R[pending[0]]["viewset"]["pending"][0])

선택 대기: ['2023년 전남 신안군 외국인주민 전체가 몇 명인지 찾아줘.']


**opt-1** 시군구별 연령별  외국인 주민현황(남) `DT_110025_A031_A` · 지방자치단체외국인주민현황

    시점=2023 | 지역=전라남도 | 지표=외국인주민현황 | 단위=명 | 거주외국인별=외국인근로자: 23848.0 명   다른 표: DT_110025_A033_A=25625.0명
    시점=2023 | 지역=전라남도 | 지표=외국인주민현황 | 단위=명 | 거주외국인별=기타외국인: 12996.0 명   다른 표: DT_110025_A033_A=22911.0명
    ※ [통계표] 매년 11월1일 기준
    ※ [통계표] *은 5인 미만인 경우임
    기준: {'조사': '지방자치단체외국인주민현황', '주기': 'Y', '기간': '2023~2023', '단위': '명', '항목': ['외국인 주민현황'], '검색 순위': 1}


**opt-2** 읍면동별 유형 및 지역별 외국인 주민현황 `DT_110025_A033_A` · 지방자치단체외국인주민현황

    시점=2023 | 지역=전라남도 | 지표=외국인주민현황 | 단위=명 | 거주외국인별=외국인근로자: 25625.0 명   다른 표: DT_110025_A031_A=23848.0명
    시점=2023 | 지역=전라남도 | 지표=외국인주민현황 | 단위=명 | 거주외국인별=기타외국인: 22911.0 명   다른 표: DT_110025_A031_A=12996.0명
    ※ [통계표] 매년 11월1일 기준
    ※ [통계표] *** & *은 5인 미만인 경우임
    ※ [자료코드 외국인주민자녀(출생)] 외국인 또는 귀화한 자의 자녀로서 국적법 제2조(출생에 의한 국적취득)에 따른 출생과 동시에 한국 국적을 취득한 자
    기준: {'조사': '지방자치단체외국인주민현황', '주기': 'Y', '기간': '2023~2023', '단위': '명', '항목': ['외국인 주민현황'], '검색 순위': 3}


In [19]:
if pending:
    choose(R, pending[0], "opt-1", CFG)          # ← 원하는 opt-n 으로
summary_table(R)

반영: [('ad71c11e-2', 'opt-1', '뺀 View', ['view::110_DT_110025_A033_A::Y::ad71c11e-2-3::value'])]


**`view::101_DT_1IN1503::Y::ad71c11e-1-1::value`**  
인구총조사 · 단위  · 기간 2023~2023 · value · 값 2칸 · [KOSIS](https://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1IN1503)

,시점,행정구역별(읍면동),연령별,항목,값
0,2023,전라남도,합계,총인구(명),1776668.0
1,2023,전라남도,합계,총인구_성비,103.5


**`view::101_DT_1B040A3::Y::ad71c11e-1-2::value`**  
주민등록인구현황 · 단위 명 · 기간 2023~2023 · value · 값 1칸 · [KOSIS](https://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1B040A3)

,시점,행정구역(시군구)별,항목,값
0,2023,전라남도,총인구수,1804217.0


**`view::101_DT_1BPB002::Y::ad71c11e-1-3::value`**  
장래인구추계 · 단위  · 기간 2023~2023 · value · 값 9칸 · [KOSIS](https://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1BPB002)

... (2 outputs left out)

,시점,행정구역(시군구)별,거주외국인별,연령별,항목,값
0,2023,전라남도,외국인근로자,계,외국인 주민현황,23848.0
1,2023,전라남도,기타외국인,계,외국인 주민현황,12996.0


**`view::110_DT_110025_A032_A::Y::ad71c11e-2-2::value`**  
지방자치단체외국인주민현황 · 단위 명 · 기간 2023~2023 · value · 값 2칸 · [KOSIS](https://kosis.kr/statHtml/statHtml.do?orgId=110&tblId=DT_110025_A032_A)

,시점,행정구역(시군구)별,거주외국인별,연령별,항목,값
0,2023,전라남도,외국인근로자,계,외국인 주민현항,1777.0
1,2023,전라남도,기타외국인,계,외국인 주민현항,9915.0


,질문,① 분석,검색 문장,단위 수,② baseline 1위,④ 충돌,최종 상태,최종 View 수,초
0,2025년 전국에서 혼자 사는 가구는 몇 가구로 집계됐지?,ok,1인가구 가구수,1,DT_1JC1516,ok,ok,3,2.66
1,2024년에서 2025년 사이 다문화가구는 전국적으로 몇 가구 늘었나?,ok,다문화가구 가구수 증감,2,DT_1JU1507 / DT_1JD1505,ok / ok,ok,6,3.37
2,2016년과 2025년의 전국 한부모가구 차이를 가구 수로 말해 줘.,ok,한부모 가구수 증감 부부+한부모,2,DT_1JU1507 / DT_1JC1517,ok / ok,ok,6,3.34
3,2025년 전국 주택은 모두 몇 호인지 알려줘.,ok,주택수,1,DT_1JU1503,ok,ok,3,2.53
4,2025년에 태어난 신생아가 전국팔도에서 몇 명이었어?,ok,영유아 출생아수 지역별 인구 영유아_계,2,DT_1B81A28 / DT_1YC1601,ok / ok,ok,6,4.05
5,2025 혼인껀수랑 2024 혼인 건수 차이,ok,혼인건수 증감 추이,1,DT_1B8000G,ok,ok,3,3.08
6,2023년 남녀 평균초혼연령 차이는 몇 살인지 알려줘.,ok,남자 여자 평균초혼연령 비교,1,DT_1B83A09,partial,ok,3,2.65
7,2023년 전남 신안군 외국인주민 전체가 몇 명인지 찾아줘.,ok,외국인주민 인구수,2,DT_1IN1503 / DT_110025_A031_A,ok / ambiguous→opt-1,ok,5,3.24
8,2025년 국내 등록외국인 가운데 베트남 국적은 몇 명이야?,ok,베트남 등록외국인 인구수,2,DT_1IN1503 / DT_1B040A7,ok / ok,ok,6,3.46
9,2026년 9월 한 달간 전국에서 이사한 사람 수는?,ok (되묻기 응답),이동자수,1,DT_1B26002,partial,ok,3,3.61


## ⑤ (선택) 답변까지 — HCX 1회

`RUN_ANSWER = True` 일 때만. 최종 View 로 `answer_from_records` (= `POST /api/v1/agent/answer`).

In [20]:
if RUN_ANSWER and not USE_FAKE:
    from IPython.display import Markdown
    from backend.agent.light_agent import answer_from_records
    for q, r in R.items():
        vs = r.get("viewset") or {}
        if vs.get("status") == "ok" and vs.get("records"):
            r["answer"] = out = answer_from_records(q, vs["records"])
            display(Markdown(f"**{q}**\n\n{out.get('answer')}\n\n_검증 안 된 숫자: {out.get('unverified_numbers')}_"))
else:
    print("RUN_ANSWER=False (또는 가짜 데이터) — 건너뜀")

**2025년 전국에서 혼자 사는 가구는 몇 가구로 집계됐지?**

2025년 전국에서 혼자 사는 가구의 수는 약 8,244,308가구로 집계되었습니다.

_검증 안 된 숫자: []_

**2024년에서 2025년 사이 다문화가구는 전국적으로 몇 가구 늘었나?**

2024년에서 2025년 사이에 전국적으로 다문화가구의 수는 약 12,188가구 증가했습니다.

_검증 안 된 숫자: ['12,188']_

**2016년과 2025년의 전국 한부모가구 차이를 가구 수로 말해 줘.**

2016년과 2025년의 전국 한부모가구 차이는 약 29,468가구 감소했습니다.

_검증 안 된 숫자: ['29,468']_

**2025년 전국 주택은 모두 몇 호인지 알려줘.**

2025년 전국의 주택 수는 약 23,239,118호입니다.

_검증 안 된 숫자: []_

**2025년에 태어난 신생아가 전국팔도에서 몇 명이었어?**

2025년 전국에서 태어난 신생아의 수는 약 254,341명으로 나타납니다.

_검증 안 된 숫자: []_

**2025 혼인껀수랑 2024 혼인 건수 차이**

2024년과 2025년의 혼인 건수는 각각 222,412건과 240,326건으로 나타났습니다.

_검증 안 된 숫자: []_

... (5 outputs left out)

**2026년 전체 인구에서 65세 이상이 차지하는 비중은 몇 퍼센트로 예상돼?**

2026년 전체 인구에서 65세 이상이 차지하는 비중은 약 21.6%로 예상됩니다.

_검증 안 된 숫자: []_

**2035년 한 집에서 솔로 가구는 얼마나 될 것으로 예상돼?**

현재 제공된 자료로는 2035년의 한 집에서의 솔로 가구 수를 예측하기 어렵습니다.

_검증 안 된 숫자: ['2035']_

**2024년 해외에서 들어온 한국인과 해외로 나간 한국인의 차이가 얼마야?**

현재 제공된 자료에서는 2024년 해외에서 들어온 한국인과 해외로 나간 한국인의 차이에 대한 정보를 찾을 수 없습니다.

_검증 안 된 숫자: []_

**2026년 9월 말 전국 주민등록 인구가 얼마로 집계됐어?**

현재 제공된 자료에서는 2026년 9월 말 전국 주민등록 인구에 대한 정보를 찾을 수 없습니다.

_검증 안 된 숫자: ['2026']_

## 저장 · 검색 구성 비교 보고서

- `R` 전체를 `storage/runs/<시각>_nb_e2e/results.json` 으로 (나중에 다시 보거나 팀원과 공유)
- 이 질문들의 검색 단위로 Step 3 비교 보고서(`report.html`)·판정표(`review.csv`)도 만든다

In [ ]:
out_dir = Path(RUNS_DIR) / f"{time.strftime('%Y%m%d_%H%M%S')}_nb_e2e"
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "results.json").write_text(json.dumps(R, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
summary_table(R).to_csv(out_dir / "summary.csv", index=False, encoding="utf-8-sig")
print("저장:", out_dir)

from backend.rag.retrieval import lab
qs = [{"id": f"Q{i}-{u['unit_id']}", "question": q, "search_text": u["search_query"], "filters": u["filters"], "prep": False}
      for i, (q, r) in enumerate(R.items(), 1) for u in r.get("retrieval") or []]
if qs:
    cmp = lab.run_compare(lab.load_queries(qs), ok_configs(CFG["retrieval_configs"]), top_k=5)
    print("비교 보고서:", cmp["files"]["html"], "\n판정표:", cmp["files"]["review"])
    display(pd.DataFrame(cmp["summary"]).T)